In [1]:
import sys
sys.path.append("/global/cfs/projectdirs/m4491/btaczak/SOLPS-BOUT-coupling/scripts")
import os
import sys
import yaml
import numpy as np
from pathlib import Path
import re
import numpy as np
import netCDF4 as nc
import yaml
from scipy.interpolate import interp1d
from boutdata.data import BoutData
from boututils.datafile import DataFile
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FormatStrFormatter
from boutdata.data import BoutData
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, fixed
from sibil_io_functions import load_config, sibil_config_summary, confirm, parse_tdata2, parse_block_sizes, write_updated_namelist, _format_value, extract_tdata_first_index, interpolate_dataset, get_midplane_distances_from_psin, vec_summary, set_tdata2_block, get_bout_midplane_distances
from converged_profile_functions import unbiased_convergence, end_biased_convergence

DEFAULT_CONFIG_PATH = "SIBIL_config.yaml"

config_path = DEFAULT_CONFIG_PATH

if not os.path.exists(config_path):
    print(f"Error: config file not found: {config_path}")
    sys.exit(1)

config = load_config(config_path)
sibil_config_summary(config)

# if not confirm():
#     print("Aborted. No changes made.")
#     sys.exit(0)

import sibil  # imported here so summary/confirmation works without dependency installed

INPUT_FILE  = f"{config["base_path"]}/{config["iteration"]}_iteration/b2.transport.inputfile"
OUTPUT_FILE = f"{config["base_path"]}/{int(config["iteration"])+1}_iteration/b2.transport.inputfile"
bout_dir  = Path(f"{config["base_path"]}/{config["iteration"]}_iteration/{config["files"]["bout_dir"]}")
data_dir  = bout_dir / "numpy"
grid_fname= Path(f"{config["base_path"]}/{config["iteration"]}_iteration/{config["files"]["bgrid"]}")
grid_fname = str(grid_fname)
transport_coeffs = config["bout_to_solps_pass"]["passed_coeffs"]
profile_sample_timestep = config["bout_to_solps_pass"]["profile_sample_timestep"]
jomp = config["bout_to_solps_pass"]["jomp"]
N, l = config["bout_to_solps_pass"]["N"], config["bout_to_solps_pass"]["l"]

grid = DataFile(grid_fname)
try:
    _ = grid["psi_bdry"]
    has_psi_bdry = True
except Exception:
    has_psi_bdry = False
    
if has_psi_bdry:
    psi_bndry = grid["psi_bdry"]
else:
    psi_bndry = grid["psi_bndry"]

psin = ((grid["psixy"] - grid["psi_axis"]) / (psi_bndry - grid["psi_axis"]))[:,jomp]

sepx_dists_solps = extract_tdata_first_index(INPUT_FILE)
psin_mid, sepx_dists_bout = get_bout_midplane_distances(grid_fname)

convergent_transport_coeffs = {}
interplted_transport_coeffs = {}
# "ddr_Tetot_txy", "ddr_Titot_txy", 
#                                        "ddr_nitot_txy", 
transport_coeffs = transport_coeffs + [#"Vr_t",
                                       "ddr_Tetot_txy", "ddr_Titot_txy", 
                                       "ddr_nitot_txy", 
                                       #"Vexbx_txy", "Vbtildx_txy"
]
for var_name in transport_coeffs:
    var = np.load(f"{data_dir}/{var_name}.npy")
    var_conv = unbiased_convergence(var, N, l)
    var_conv_midplane = var_conv[profile_sample_timestep,:,jomp]
    var_conv_interpolated = interpolate_dataset(sepx_dists_bout,var_conv_midplane,sepx_dists_solps)
    nan_mask = np.isnan(var_conv_interpolated)
    var_conv_interpolated[nan_mask]=0
    
    convergent_transport_coeffs[var_name] = var_conv
    interplted_transport_coeffs[var_name] = var_conv_interpolated

SIBIL Midplane Mapper — Configuration Summary
  G-file         : ../g184016.03200  [OK]
  Geometry file  : ../baserun/HvsD_Dmesh_noC_v1.v001.geo  [OK]
  B-grid file    : r9.2_fr8.0_flat_pfr/d3d_shelf_184016.03506_436_nc3_jpar_repair_only_pfflat.grd.nc  [OK]
  b2fstate       : b2fstate  [OK]
  b2fplasmf      : b2fplasmf  [OK]
  b2fgmtry       : b2fgmtry  [OK]

  Softmax scales (max, min):
    ni  : max=1e+16, min=1e+15
    ne  : max=1e+16, min=1e+15
    Ti  : max=0.5, min=0.0001
    Te  : max=0.5, min=0.0001
SIBIL Succsessfully Imported


FileNotFoundError: [Errno 2] No such file or directory: '/global/cfs/projectdirs/m4491/btaczak/SIBIL-runs/DIIID_LSN_noC_kefit_base/0_iteration/r9.2_fr8.0_flat_pfr/numpy/Da_t.npy'

In [ ]:
import sibil_debug_diagnostics as dbg
display(dbg.make_debug_widget(convergent_transport_coeffs, psin_mid,
                              jomp=jomp, grid=grid, data_dir=data_dir))
dbg.full_report(convergent_transport_coeffs, psin_mid, grid_fname,
                jomp=jomp, savedir="./crash_diagnostics")